# 01 – Raw Data: Dataset Original

**Proyecto:** Predicción de subempleo por insuficiencia de horas en trabajadores ocupados de Lima Metropolitana  
**Objetivo del modelo:** Predecir la probabilidad de que un trabajador ocupado presente subempleo por insuficiencia de horas usando variables sociodemográficas, educativas, laborales, de ingresos y protección social.  
**Target:** `P209H` recodificada como binaria → `1` = tiene voluntad y disponibilidad de trabajar más horas · `0` = no la tiene

## 1. Descripción de la fuente de datos

- **Fuente:** Encuesta Permanente de Empleo Nacional (EPEN) 2024
- **Institución:** Instituto Nacional de Estadística e Informática (INEI) – Perú
- **Archivo:** `250806 EPEN2024.xlsx`
- **Período de referencia:** Trimestre Sep-Oct-Nov 2024
- **Unidad de análisis:** Personas residentes de **14 años a más, ocupadas, en Lima Metropolitana** (`REGION == 1`)
- **Cobertura geográfica:** Nacional (filtrado a Lima Metropolitana)

---

## 2. Variable objetivo (target)

| Variable | Pregunta original | Recodificación |
|---|---|---|
| `P209H` | ¿TUVO LA VOLUNTAD DE TRABAJAR MÁS HORAS Y ADEMÁS ESTUVO DISPONIBLE PARA HACERLO? | `1` = Sí (subempleado por horas) · `0` = No |

> Corresponde a la combinación de `C333` (quería trabajar más horas) y `C334` (estuvo disponible para hacerlo).

---

## 3. Variables del dataset (132 columnas según diccionario INEI)

### Identificación y diseño muestral
| Variable | Descripción |
|---|---|
| `ANIO` | Año de la encuesta |
| `MES` | Mes de la encuesta |
| `CONGLOMERADO` | Número del conglomerado |
| `MUESTRA` | N° de sub muestra |
| `REGION` | Región (1=Lima Met. · 2=Resto urbano · 3=Rural) |
| `ESTRATO` | Estrato geográfico (1–8) |
| `LLAVE_PANEL` | Código de persona panel |

### Características sociodemográficas
| Variable | Descripción |
|---|---|
| `C201` | N° de orden / código de persona |
| `C203` | Relación de parentesco con el jefe del hogar (1=Jefe … 11=Otro no pariente) |
| `C207` | Sexo (1=Hombre · 2=Mujer) |
| `C208` | Edad en años cumplidos |
| `C301_DIA/MES/ANIO` | Fecha de nacimiento |

### Condición de actividad (módulo 300)
| Variable | Descripción |
|---|---|
| `C303` | La semana pasada, ¿tuvo algún trabajo? (1=Sí · 2=No) |
| `C304` | ¿Tiene empleo fijo al que próximamente volverá? |
| `C305` | ¿Tiene negocio propio al que próximamente volverá? |
| `C306_1…C306_11` | Actividades realizadas ≥1 hora para obtener ingresos |

### Ocupación principal
| Variable | Descripción |
|---|---|
| `C308_COD` | Código de ocupación principal (CNO) |
| `C309_COD` | Código de actividad económica de la empresa (CIIU) |
| `C310` | Categoría ocupacional (1=Empleador · 2=Independiente · 3=Empleado/obrero · 4–10=otras) |
| `C311` | Tipo de empleador (1=FF.AA./PNP · 2=Admin. pública · 3=Empresa pública · 4=Service · 5=Empresa privada · 6=Otra) |
| `C312` | Registro en SUNAT (1=Persona jurídica · 2=Natural con RUC · 3=No registrado · 4=No sabe) |
| `C313` | Lleva libros contables (1=Sí · 2=No · 3=No sabe) |
| `C317` | Tamaño de empresa (1=≤20 · 2=21–50 · …) |

### Horas trabajadas
| Variable | Descripción |
|---|---|
| `C318_1…C318_7` | Horas trabajadas cada día (lunes a sábado) en ocupación principal |
| `C318_T` | Total horas trabajadas en ocupación principal |
| `C328_T` | Horas trabajadas en ocupaciones secundarias |
| `whoraT` | Horas totales (todas las ocupaciones) |
| `C330` | ¿Normalmente trabaja esas horas? (1=Sí · 2=No) |
| `C331` | Horas normales semanales en todas las ocupaciones |

### Subempleo por horas (origen del target)
| Variable | Descripción |
|---|---|
| `C333` | ¿La semana pasada quería trabajar más horas? (1=Sí · 2=No) |
| `C334` | ¿Estuvo disponible para trabajar más horas? (1=Sí · 2=No) |
| **`P209H`** | **¿Tuvo voluntad Y disponibilidad de trabajar más horas? (TARGET)** |

### Protección social
| Variable | Descripción |
|---|---|
| `SEGURO1` | Indicador de afiliación a algún seguro de salud |
| `C361_1…C361_8` | Tipos de seguro de salud (SIS, EsSalud, privado, FFAA, etc.) |
| `C364_1…C364_4` | Tipos de sistema de pensiones (AFP, SNP, etc.) |

### Ingresos
| Variable | Descripción |
|---|---|
| `C339_1` | Ingreso en ocupación principal (sin descuentos) |
| `C341_T` | Total ingresos ocupación principal |
| `INGTOT` | Ingreso total mensual |
| `INGTOTP` | Ingreso total per cápita |
| `ingtrabw` | Ingreso laboral (winsorizado) |

### Factores de expansión y otros
| Variable | Descripción |
|---|---|
| `OCUP300` | Ocupación principal según clasificador |
| `RESIDENT` | Condición de residencia |
| `fa_ond24 / fa_efm24 / fa_amj24 / fa_jas24` | Factor de expansión por trimestre |

In [ ]:
# ─── Librerías ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os

In [ ]:
# ─── Rutas ────────────────────────────────────────────────────────────────────
BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
DATA_PATH = os.path.join(BASE_DIR, '250806 EPEN2024.xlsx')

# ─── Carga del dataset completo (raw, sin modificaciones) ─────────────────────
df_raw = pd.read_excel(DATA_PATH)

print(f'Dataset completo cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'Columnas: {df_raw.columns.tolist()}')
df_raw.head()

Dataset cargado: 1,000 filas × 9 columnas


,edad,sexo,nivel_educativo,estado_civil,ingreso_mensual,horas_trabajadas,tipo_empleo,sector,condicion_actividad
0,52,Mujer,Posgrado,Viudo,14103.94,39,Formal,Comercio,No PEA
1,65,Mujer,Universidad,Casado,2792.14,10,Informal,Industria,Ocupado
2,42,Mujer,Secundaria,Unión libre,115.35,45,nan,Comercio,Ocupado
3,28,Mujer,Universidad,Viudo,10711.30,7,Informal,Industria,Ocupado
4,56,Mujer,Posgrado,Viudo,14373.11,28,Formal,Gobierno,No PEA


## 4. Universo analítico (solo referencia – sin modificar raw)

Los filtros que se aplicarán en el paso de preprocesamiento son:

1. `REGION == 1` → Lima Metropolitana  
2. `C208 >= 14` → Personas de 14 años a más  
3. `RESIDENT == 1` → Residentes habituales del hogar  
4. Condición de ocupado (definida a partir de `C303`, `C304`, `C305`, `C306_*`)  
5. `P209H` no nulo → tiene información sobre el target  

A continuación se muestra un conteo preliminar **sin aplicar filtros** para entender la distribución.

In [ ]:
# ─── Distribución preliminar por REGION ──────────────────────────────────────
print('=== Distribución por REGION ===')
print(df_raw['REGION'].value_counts().rename({1: 'Lima Metropolitana', 2: 'Resto urbano', 3: 'Rural'}))

print('\n=== Rango de edad (C208) ===')
print(df_raw['C208'].describe())

print('\n=== Target P209H (dataset completo) ===')
print(df_raw['P209H'].value_counts(dropna=False))

# Conteo anticipado del universo analítico
mask_lima    = df_raw['REGION'] == 1
mask_edad    = df_raw['C208'] >= 14
mask_target  = df_raw['P209H'].notna()
n_universo   = (mask_lima & mask_edad & mask_target).sum()
print(f'\nRegistros que cumplen REGION=1 + edad≥14 + P209H no nulo: {n_universo:,}')

In [3]:
# ─── Información general ──────────────────────────────────────────────────────
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   edad                 1000 non-null   int64  
 1   sexo                 1000 non-null   str    
 2   nivel_educativo      1000 non-null   str    
 3   estado_civil         1000 non-null   str    
 4   ingreso_mensual      1000 non-null   float64
 5   horas_trabajadas     1000 non-null   int64  
 6   tipo_empleo          1000 non-null   str    
 7   sector               1000 non-null   str    
 8   condicion_actividad  1000 non-null   str    
dtypes: float64(1), int64(2), str(6)
memory usage: 70.4 KB


In [4]:
# ─── Estadísticas descriptivas ────────────────────────────────────────────────
df.describe(include='all')

,edad,sexo,nivel_educativo,estado_civil,ingreso_mensual,horas_trabajadas,tipo_empleo,sector,condicion_actividad
count,1000.000000,1000,1000,1000,1000.000000,1000.000000,1000,1000,1000
unique,NaN,2,6,5,NaN,NaN,3,5,3
top,NaN,Hombre,Preparatoria,Divorciado,NaN,NaN,Informal,Industria,Ocupado
freq,NaN,519,179,212,NaN,NaN,466,231,622
mean,41.852000,NaN,NaN,NaN,7952.472260,29.791000,NaN,NaN,NaN
std,16.069796,NaN,NaN,NaN,7838.114053,17.386075,NaN,NaN,NaN
min,14.000000,NaN,NaN,NaN,0.090000,0.000000,NaN,NaN,NaN
25%,28.750000,NaN,NaN,NaN,2289.225000,15.000000,NaN,NaN,NaN
50%,42.000000,NaN,NaN,NaN,5629.060000,30.000000,NaN,NaN,NaN
75%,55.000000,NaN,NaN,NaN,11057.750000,45.000000,NaN,NaN,NaN


In [ ]:
# ─── Guardar snapshot del raw data (sin ninguna modificación) ─────────────────
SNAPSHOT_DIR  = os.path.join(BASE_DIR, '..', 'data', 'raw')
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
SNAPSHOT_PATH = os.path.join(SNAPSHOT_DIR, 'epen2024_raw_snapshot.csv')
df_raw.to_csv(SNAPSHOT_PATH, index=False, encoding='utf-8-sig')
print(f'Snapshot raw guardado: {os.path.abspath(SNAPSHOT_PATH)}')
print(f'Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}')

Snapshot guardado correctamente.
